In [ ]:
!pip install -U transformers --upgrade

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!fusermount -u /content/drive
!rm -rf /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
!fusermount -u /content/drive
!rm -rf /content/drive


In [ ]:
!pip install -q pypdf
!pip install -q python-dotenv
!pip install  llama-index==0.10.12
!pip install -q gradio
!pip install einops
!pip install accelerate

In [ ]:
!pip install llama-index-llms-huggingface

In [ ]:
!pip install llama-index-embeddings-fastembed

In [ ]:
!pip install fastembed

In [ ]:
import logging
import sys
!pip uninstall -y numpy transformers
!pip install --upgrade numpy transformers

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings

documents = SimpleDirectoryReader("/content/drive/MyDrive/Data").load_data()


In [ ]:
from llama_index.embeddings.fastembed import FastEmbedEmbedding

embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model
Settings.chunk_size = 512

In [ ]:
from llama_index.core import PromptTemplate


system_prompt = """
Answer the questions according to the Indian philosophical context, drawing insights from the following three sources:

1. Bhagavad Gita – Explain using Lord Krishna's teachings from the Gita.
2. Vedas – Provide insights from Vedic scriptures.
3. Astrology – Explain the astrological interpretation.

Ensure each response is structured into these three sections. Cite relevant references where applicable.
If context is not available, give a general response.

"""



query_wrapper_prompt = PromptTemplate("<|USER|>{query_str}<|ASSISTANT|>")




In [ ]:
# @title .
from huggingface_hub import  notebook_login
notebook_login()

In [ ]:
# @title .

import torch

llm = HuggingFaceLLM(
    context_window=8192,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.7, "do_sample": False},
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="google/gemma-7b-it",
    model_name="google/gemma-7b-it",
    device_map="auto",
    tokenizer_kwargs={"max_length": 4096},
    model_kwargs={"torch_dtype": torch.float16}
)

Settings.llm = llm
Settings.chunk_size = 512

In [ ]:
index = VectorStoreIndex.from_documents(documents)


In [ ]:

query_engine = index.as_query_engine()

def predict(input, history):
  response = query_engine.query(input)
  return str(response)


In [ ]:
import gradio as gr
gr.ChatInterface(predict).launch(share=True)

In [ ]:
import csv
import os
import openpyxl
import gradio as gr
from transformers import pipeline

drive_path = "/content/drive/MyDrive/"
ratings_csv_file = os.path.join(drive_path, "ratings.csv")
ratings_excel_file = os.path.join(drive_path, "ratings.xlsx")

translator_en_hi = pipeline("translation", model="Helsinki-NLP/opus-mt-en-hi")
translator_hi_en = pipeline("translation", model="Helsinki-NLP/opus-mt-hi-en")

try:
    documents = SimpleDirectoryReader(os.path.join(drive_path, "Data")).load_data()
    print(f" Documents Loaded: {len(documents)}")
    index = VectorStoreIndex.from_documents(documents)
    print(" Index initialized successfully.")
except FileNotFoundError:
    print(" Error: Document directory not found.")
    index = None
except Exception as e:
    print(f" Error initializing index: {str(e)}")
    index = None

#  CSV
def save_rating_csv(question, answer, rating):
    if not os.path.exists(ratings_csv_file):
        with open(ratings_csv_file, mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["Question", "Answer", "Rating"])

    with open(ratings_csv_file, mode="a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([question, answer, rating])

def save_rating_excel(question, answer, rating):
    if not os.path.exists(ratings_excel_file):
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.append(["Question", "Answer", "Rating"])
        wb.save(ratings1_excel_file)

    wb = openpyxl.load_workbook(ratings_excel_file)
    ws = wb.active
    ws.append([question, answer, rating])
    wb.save(ratings1_excel_file)

def save_rating(question, answer, rating):
    save_rating_csv(question, answer, rating)
    save_rating_excel(question, answer, rating)

def predict(input_text, selected_language, index):
    if index is None:
        return "⚠ Error: Index is not initialized!"

    if any("\u0900" <= char <= "\u097F" for char in input_text):
        input_text = translator_hi_en(input_text)[0]["translation_text"]

    try:
        query_engine = index.as_query_engine()
        response = query_engine.query(input_text)
        response_text = str(response)
    except Exception as e:
        return f" Error processing query: {str(e)}"

    if selected_language == "Hindi":
        response_text = translator_en_hi(response_text)[0]["translation_text"]

    return response_text

#  Gradio
def chat_interface_with_rating(index):
    with gr.Blocks() as demo:
        gr.Markdown("# Vedic AI")
        gr.Markdown("Ask questions in *English or Hindi*, and choose response language.")

        input_text = gr.Textbox(label="Ask your question")
        selected_language = gr.Dropdown(["English", "Hindi"], label="Choose Response Language", value="English")
        output_text = gr.Textbox(label="Chatbot Response")

        submit_button = gr.Button("Get Answer")
        submit_button.click(lambda text, lang: predict(text, lang, index),
                            inputs=[input_text, selected_language],
                            outputs=output_text)


        rating = gr.Dropdown(["1", "2", "3", "4", "5"], label="Rate the Response (Optional)", value=None)
        rate_button = gr.Button("Submit Rating")


        rate_button.click(lambda q, a, r: save_rating(q, a, r) if r else "No rating given",
                          inputs=[input_text, output_text, rating],
                          outputs=None)

        demo.launch(share=True)


chat_interface_with_rating(index)


In [ ]:
!pip install rouge-score

from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer

expected_response = "Your success and life path depend on your personal choices, efforts, and circumstances. People born under Aquarius tend to be independent thinkers, innovative, and driven to contribute to society. They may excel in research, technology, or humanitarian fields. Developing resilience and adaptability can lead to success."
vedic_ai_response = "Dhanishta Nakshatra in Kumbha Raashi is linked to prosperity, adaptability, and musical talent. Individuals under this nakshatra are hardworking, resourceful, and capable of great success. Aquarius influence fosters a scientific and inquisitive mindset, encouraging unconventional paths. Cultivating patience and balancing personal goals with societal contributions will lead to a fulfilling life."

# BLEU Calculation
bleu_score = sentence_bleu([expected_response.split()], vedic_ai_response.split())
print(f"BLEU Score: {bleu_score:.4f}")

# ROUGE Calculation
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
rouge_scores = scorer.score(expected_response, vedic_ai_response)

print(f"ROUGE-1 Score: {rouge_scores['rouge1'].fmeasure:.4f}")
print(f"ROUGE-L Score: {rouge_scores['rougeL'].fmeasure:.4f}")
